In [1]:
from typing import Optional, List
from sqlalchemy.orm import Mapped, mapped_column, relationship, Session, DeclarativeBase
from sqlalchemy import ForeignKey, select, exists
from sqlalchemy import Table, Column, Integer, String
from sqlalchemy import create_engine

In [2]:
engine = create_engine("sqlite+pysqlite:///:memory:", echo=True)

In [3]:
class Base(DeclarativeBase):
    pass


class User(Base):
    __tablename__ = "user_account"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(30))
    fullname: Mapped[Optional[str]]

    addresses: Mapped[List["Address"]] = relationship(back_populates="user")

    def __repr__(self) -> str:
        return f"User(id={self.id!r}, name={self.name!r}, fullname={self.fullname!r})"


class Address(Base):
    __tablename__ = "address"

    id: Mapped[int] = mapped_column(primary_key=True)
    email_address: Mapped[str]
    user_id = mapped_column(ForeignKey("user_account.id"))

    user: Mapped[User] = relationship(back_populates="addresses")

    def __repr__(self) -> str:
        return f"Address(id={self.id!r}, email_address={self.email_address!r})"

In [4]:
sandy = User(name="sandy", fullname="Sandy Cheeks")
spongebob = User(name="spongebob", fullname="Spongebob Squarepants")
patrick = User(name="patrick", fullname="Patrick Star")

In [5]:
Base.metadata.create_all(engine)

2025-04-22 17:10:39,947 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:10:39,949 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("user_account")
2025-04-22 17:10:39,950 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-04-22 17:10:39,952 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("user_account")
2025-04-22 17:10:39,952 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-04-22 17:10:39,953 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("address")
2025-04-22 17:10:39,953 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-04-22 17:10:39,954 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("address")
2025-04-22 17:10:39,954 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-04-22 17:10:39,956 INFO sqlalchemy.engine.Engine 
CREATE TABLE user_account (
	id INTEGER NOT NULL, 
	name VARCHAR(30) NOT NULL, 
	fullname VARCHAR, 
	PRIMARY KEY (id)
)


2025-04-22 17:10:39,957 INFO sqlalchemy.engine.Engine [no key 0.00090s] ()
2025-04-22 17:10:39,958 INFO sqlalchemy.engine.

In [6]:
def get_session(engine):
    return Session(engine)

In [7]:
with get_session(engine) as session:
    session.add(sandy)
    session.add(spongebob)
    session.add(patrick)
    session.commit()

2025-04-22 17:10:39,970 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:10:39,972 INFO sqlalchemy.engine.Engine INSERT INTO user_account (name, fullname) VALUES (?, ?) RETURNING id
2025-04-22 17:10:39,973 INFO sqlalchemy.engine.Engine [generated in 0.00012s (insertmanyvalues) 1/3 (ordered; batch not supported)] ('sandy', 'Sandy Cheeks')
2025-04-22 17:10:39,973 INFO sqlalchemy.engine.Engine INSERT INTO user_account (name, fullname) VALUES (?, ?) RETURNING id
2025-04-22 17:10:39,974 INFO sqlalchemy.engine.Engine [insertmanyvalues 2/3 (ordered; batch not supported)] ('spongebob', 'Spongebob Squarepants')
2025-04-22 17:10:39,974 INFO sqlalchemy.engine.Engine INSERT INTO user_account (name, fullname) VALUES (?, ?) RETURNING id
2025-04-22 17:10:39,974 INFO sqlalchemy.engine.Engine [insertmanyvalues 3/3 (ordered; batch not supported)] ('patrick', 'Patrick Star')
2025-04-22 17:10:39,976 INFO sqlalchemy.engine.Engine COMMIT


In [8]:
with get_session(engine) as session:
    result = session.execute(select(User)).all()
    print(result)

2025-04-22 17:10:39,983 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:10:39,987 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account
2025-04-22 17:10:39,987 INFO sqlalchemy.engine.Engine [generated in 0.00047s] ()
[(User(id=1, name='sandy', fullname='Sandy Cheeks'),), (User(id=2, name='spongebob', fullname='Spongebob Squarepants'),), (User(id=3, name='patrick', fullname='Patrick Star'),)]
2025-04-22 17:10:39,989 INFO sqlalchemy.engine.Engine ROLLBACK


In [9]:
with get_session(engine) as session:
    result = session.scalar(select(exists(User).where(User.name == "patrick")))
    # stmt = select(exists().where(User.name == 'some_value'))
    # result = session.execute(stmt).scalar()
    print(result)

2025-04-22 17:10:39,999 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:10:40,001 INFO sqlalchemy.engine.Engine SELECT EXISTS (SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account 
WHERE user_account.name = ?) AS anon_1
2025-04-22 17:10:40,003 INFO sqlalchemy.engine.Engine [generated in 0.00156s] ('patrick',)
True
2025-04-22 17:10:40,005 INFO sqlalchemy.engine.Engine ROLLBACK


In [10]:
with get_session(engine) as session:
    result = session.scalar(select(User.id).where(User.name == "patrdsfrick"))
    print(result)

2025-04-22 17:10:40,011 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:10:40,013 INFO sqlalchemy.engine.Engine SELECT user_account.id 
FROM user_account 
WHERE user_account.name = ?
2025-04-22 17:10:40,013 INFO sqlalchemy.engine.Engine [generated in 0.00073s] ('patrdsfrick',)
None
2025-04-22 17:10:40,014 INFO sqlalchemy.engine.Engine ROLLBACK


In [11]:
with get_session(engine) as session:
    result = session.scalars(select(User)).all()
    print(result)

2025-04-22 17:10:40,019 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:10:40,020 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account
2025-04-22 17:10:40,020 INFO sqlalchemy.engine.Engine [cached since 0.03314s ago] ()
[User(id=1, name='sandy', fullname='Sandy Cheeks'), User(id=2, name='spongebob', fullname='Spongebob Squarepants'), User(id=3, name='patrick', fullname='Patrick Star')]
2025-04-22 17:10:40,021 INFO sqlalchemy.engine.Engine ROLLBACK


In [12]:
with get_session(engine) as session:
    result = session.execute(select(User.id, User.name, User.fullname)).all()
    # stmt = select(exists().where(User.name == 'some_value'))
    # result = session.execute(stmt).scalar()
    print(result)
    print([(r.id, r.name, r.fullname) for r in result])

2025-04-22 17:10:40,026 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:10:40,027 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account
2025-04-22 17:10:40,029 INFO sqlalchemy.engine.Engine [generated in 0.00187s] ()
[(1, 'sandy', 'Sandy Cheeks'), (2, 'spongebob', 'Spongebob Squarepants'), (3, 'patrick', 'Patrick Star')]
[(1, 'sandy', 'Sandy Cheeks'), (2, 'spongebob', 'Spongebob Squarepants'), (3, 'patrick', 'Patrick Star')]
2025-04-22 17:10:40,031 INFO sqlalchemy.engine.Engine ROLLBACK


In [13]:
with get_session(engine) as session:
    result = session.scalars(select(User.addresses)).all()
    print(result)

2025-04-22 17:10:40,043 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:10:40,051 INFO sqlalchemy.engine.Engine SELECT user_account.id = address.user_id AS addresses 
FROM user_account, address
2025-04-22 17:10:40,059 INFO sqlalchemy.engine.Engine [generated in 0.00866s] ()
[]
2025-04-22 17:10:40,060 INFO sqlalchemy.engine.Engine ROLLBACK


/tmp/ipykernel_165976/390227140.py:2: SAWarning: SELECT statement has a cartesian product between FROM element(s) "address" and FROM element "user_account".  Apply join condition(s) between each element to resolve.
  result = session.scalars(select(User.addresses)).all()


In [29]:
from sqlalchemy import text


with engine.connect() as conn:
        result = conn.execute(
            text("SELECT name FROM user_account") 
        )
        
        print(result.rowcount)
        for row in result.mappings():
            print(row["name"])

2025-04-22 17:18:39,511 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:18:39,512 INFO sqlalchemy.engine.Engine SELECT name FROM user_account
2025-04-22 17:18:39,512 INFO sqlalchemy.engine.Engine [cached since 304s ago] ()
-1
sandy
spongebob
patrick
2025-04-22 17:18:39,513 INFO sqlalchemy.engine.Engine ROLLBACK
